# Yoga Pose Recognition Using CNN and MobileNetV2

This notebook contains the complete implementation for yoga pose recognition using a Custom CNN and MobileNetV2 transfer learning.

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image, ImageFile

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, CSVLogger, ReduceLROnPlateau

from sklearn.metrics import classification_report, confusion_matrix

ImageFile.LOAD_TRUNCATED_IMAGES = True

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

RAW_BASE = "/kaggle/input/datasets/niharika41298/yoga-poses-dataset/DATASET"

PROJECT_DIR = "/kaggle/working/Yoga-Pose-Recognition"
FIGURE_DIR = PROJECT_DIR + "/Figures_2"
MODEL_DIR = PROJECT_DIR + "/Models"
RESULT_DIR = PROJECT_DIR + "/Results"
SOURCE_DIR = PROJECT_DIR + "/Source"
CLEAN_BASE = PROJECT_DIR + "/Clean_Dataset"

for folder in [PROJECT_DIR, FIGURE_DIR, MODEL_DIR, RESULT_DIR, SOURCE_DIR, CLEAN_BASE]:
    os.makedirs(folder, exist_ok=True)

IMG_SIZE = (160, 160)
BATCH_SIZE = 32
NUM_CLASSES = 5

class_names = ["downdog", "goddess", "plank", "tree", "warrior2"]

print("Project setup completed.")
print("Dataset path:", RAW_BASE)


# ============================================================
# Section 2: Clean Dataset Creation and Dataset Description
# ============================================================

import os
import shutil
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

clean_count = 0
bad_files = []

for split in ["TRAIN", "TEST"]:
    split_path = os.path.join(RAW_BASE, split)

    for cls in class_names:
        in_dir = os.path.join(split_path, cls)
        out_dir = os.path.join(CLEAN_BASE, split, cls)
        os.makedirs(out_dir, exist_ok=True)

        for file in os.listdir(in_dir):
            if file.lower().endswith((".jpg", ".jpeg", ".png")):
                in_path = os.path.join(in_dir, file)
                out_path = os.path.join(out_dir, os.path.splitext(file)[0] + ".jpg")

                try:
                    img = Image.open(in_path).convert("RGB")
                    img = img.resize(IMG_SIZE)
                    img.save(out_path, "JPEG", quality=95)
                    clean_count += 1
                except Exception as e:
                    bad_files.append([split, cls, file, str(e)])

print("Clean images saved:", clean_count)
print("Bad files skipped:", len(bad_files))


# ============================================================
# Section 2B: Dataset Statistics
# ============================================================

records = []

for split in ["TRAIN", "TEST"]:
    for cls in class_names:
        folder = os.path.join(CLEAN_BASE, split, cls)
        count = len([f for f in os.listdir(folder) if f.endswith(".jpg")])

        records.append({
            "split": split,
            "class": cls,
            "number_of_images": count
        })

dataset_stats = pd.DataFrame(records)

dataset_stats.to_excel(
    os.path.join(RESULT_DIR, "dataset_statistics.xlsx"),
    index=False
)

print(dataset_stats)
print("Dataset statistics saved.")


# ============================================================
# Section 3: Dataset Figures
# ============================================================

import matplotlib.pyplot as plt

# Figure: Dataset Characterization
plt.figure(figsize=(8,5))
train_stats = dataset_stats[dataset_stats["split"]=="TRAIN"]
plt.bar(train_stats["class"], train_stats["number_of_images"], color="skyblue")
plt.title("Dataset Characterization: Number of Training Images per Class")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "dataset_characterization_dashboard.pdf"), dpi=300)
plt.show()

# Figure: Sample Images per Class (5 random examples)
fig, axes = plt.subplots(5,5, figsize=(12,12))
for i, cls in enumerate(class_names):
    cls_folder = os.path.join(CLEAN_BASE, "TRAIN", cls)
    files = [f for f in os.listdir(cls_folder) if f.endswith(".jpg")]
    sample_files = np.random.choice(files, 5, replace=False)
    for j, f in enumerate(sample_files):
        img = Image.open(os.path.join(cls_folder, f))
        axes[i,j].imshow(img)
        axes[i,j].axis("off")
        if j==2:
            axes[i,j].set_title(cls, fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "representative_sample_images.pdf"), dpi=300)
plt.show()

print("Section 3 figures saved in Figures_2.")


# ============================================================
# Section 4: TensorFlow Data Pipeline and Augmentation
# ============================================================

train_data = tf.keras.utils.image_dataset_from_directory(
    CLEAN_BASE + "/TRAIN",
    validation_split=0.20,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

val_data = tf.keras.utils.image_dataset_from_directory(
    CLEAN_BASE + "/TRAIN",
    validation_split=0.20,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE
)

test_data = tf.keras.utils.image_dataset_from_directory(
    CLEAN_BASE + "/TEST",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

AUTOTUNE = tf.data.AUTOTUNE
train_data = train_data.prefetch(AUTOTUNE)
val_data = val_data.prefetch(AUTOTUNE)
test_data = test_data.prefetch(AUTOTUNE)

print("Data pipeline ready.")


# ============================================================
# Section 4B: Data Augmentation Figure
# ============================================================

augmentation = models.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.10)
])

sample_image_path = os.path.join(CLEAN_BASE, "TRAIN", "tree", os.listdir(os.path.join(CLEAN_BASE, "TRAIN", "tree"))[0])
img = Image.open(sample_image_path).convert("RGB")
img_array = np.array(img)

fig, axes = plt.subplots(1, 5, figsize=(15,4))

axes[0].imshow(img_array)
axes[0].set_title("Original")
axes[0].axis("off")

for i in range(1,5):
    aug_img = augmentation(tf.expand_dims(img_array, 0), training=True)
    axes[i].imshow(aug_img[0].numpy().astype("uint8"))
    axes[i].set_title("Augmented")
    axes[i].axis("off")

plt.suptitle("Data Augmentation Examples")
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "data_augmentation_examples.pdf"), dpi=300, bbox_inches="tight")
plt.show()

print("Data augmentation figure saved.")


# ============================================================
# Section 5: Custom CNN Model
# ============================================================

custom_cnn = models.Sequential([
    layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1],3)),
    layers.Rescaling(1./255),
    
    layers.Conv2D(32, (3,3), activation="relu", padding="same"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, (3,3), activation="relu", padding="same"),
    layers.MaxPooling2D(),
    layers.Conv2D(128, (3,3), activation="relu", padding="same"),
    layers.MaxPooling2D(),
    
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(NUM_CLASSES, activation="softmax")
])

custom_cnn.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

custom_cnn.summary()

callbacks_cnn = [
    ModelCheckpoint(
        MODEL_DIR + "/custom_cnn_best.keras",
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-6,
        verbose=1
    ),
    CSVLogger(
        RESULT_DIR + "/custom_cnn_training_log.csv",
        append=True
    )
]

# Train for 5 epochs first
history_cnn = custom_cnn.fit(
    train_data,
    validation_data=val_data,
    epochs=5,
    callbacks=callbacks_cnn
)

custom_cnn.save(MODEL_DIR + "/custom_cnn_final.keras")

print("Custom CNN training completed and saved.")


# ============================================================
# Section 6: MobileNetV2 Transfer Learning
# ============================================================

from tensorflow.keras.applications import MobileNetV2

# Load pretrained base model
base_model = MobileNetV2(
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3),
    include_top=False,
    weights="imagenet"
)
base_model.trainable = False  # freeze base

# Add custom classifier
mobilenet_model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.4),
    layers.Dense(128, activation="relu"),
    layers.Dense(NUM_CLASSES, activation="softmax")
])

mobilenet_model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

mobilenet_model.summary()

# Callbacks
callbacks_mobilenet = [
    ModelCheckpoint(
        MODEL_DIR + "/mobilenetv2_best.keras",
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    CSVLogger(
        RESULT_DIR + "/mobilenetv2_training_log.csv",
        append=True
    )
]

# Train MobileNetV2
history_mobilenet = mobilenet_model.fit(
    train_data,
    validation_data=val_data,
    epochs=15,   # you can increase to 20 for full run
    callbacks=callbacks_mobilenet
)

mobilenet_model.save(MODEL_DIR + "/mobilenetv2_final.keras")
print("MobileNetV2 training completed and saved.")


# ============================================================
# Section 7: Evaluation & Figures
# ============================================================

import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import pandas as pd

# Get true labels and predictions
y_true = np.concatenate([y for x, y in test_data], axis=0)
y_pred_prob = mobilenet_model.predict(test_data)
y_pred = np.argmax(y_pred_prob, axis=1)

# Classification report
report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
report_df = pd.DataFrame(report).transpose()
report_df.to_excel(os.path.join(RESULT_DIR, "mobilenetv2_classification_report.xlsx"))
print(report_df)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7,6))
plt.imshow(cm, cmap="Blues")
plt.title("MobileNetV2 Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks(range(len(class_names)), class_names, rotation=45)
plt.yticks(range(len(class_names)), class_names)
for i in range(len(class_names)):
    for j in range(len(class_names)):
        plt.text(j, i, cm[i,j], ha="center", va="center", color="black")
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "mobilenetv2_confusion_matrix.pdf"), dpi=300)
plt.show()

# Accuracy and loss curves
history_df = pd.DataFrame(history_mobilenet.history)
history_df.to_excel(os.path.join(RESULT_DIR, "mobilenetv2_history.xlsx"), index=False)

plt.figure(figsize=(7,5))
plt.plot(history_df["accuracy"], label="Training Accuracy")
plt.plot(history_df["val_accuracy"], label="Validation Accuracy")
plt.title("MobileNetV2 Accuracy Curve")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "mobilenetv2_accuracy_curve.pdf"), dpi=300)
plt.show()

plt.figure(figsize=(7,5))
plt.plot(history_df["loss"], label="Training Loss")
plt.plot(history_df["val_loss"], label="Validation Loss")
plt.title("MobileNetV2 Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURE_DIR, "mobilenetv2_loss_curve.pdf"), dpi=300)
plt.show()

print("Evaluation figures and reports saved successfully.")


# Representative correct and misclassified predictions

all_images = []
all_labels = []

for x, y in test_data:
    all_images.append(x.numpy())
    all_labels.append(y.numpy())

all_images = np.concatenate(all_images, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

pred_probs = mobilenet_model.predict(all_images)
pred_labels = np.argmax(pred_probs, axis=1)

correct_indices = np.where(pred_labels == all_labels)[0][:16]
wrong_indices = np.where(pred_labels != all_labels)[0][:16]

def save_prediction_grid(indices, title, filename, color):
    plt.figure(figsize=(12,12))

    for i, idx in enumerate(indices):
        plt.subplot(4,4,i+1)
        plt.imshow(all_images[idx].astype("uint8"))
        plt.axis("off")

        true_label = class_names[int(all_labels[idx])]
        pred_label = class_names[int(pred_labels[idx])]

        plt.title(
            f"True: {true_label}\nPred: {pred_label}",
            color=color,
            fontsize=9
        )

    plt.suptitle(title, fontsize=15, fontweight="bold")
    plt.tight_layout(rect=[0,0,1,0.96])
    plt.savefig(os.path.join(FIGURE_DIR, filename), dpi=300, bbox_inches="tight")
    plt.show()

save_prediction_grid(
    correct_indices,
    "Representative Correct Predictions",
    "representative_correct_predictions.pdf",
    "green"
)

save_prediction_grid(
    wrong_indices,
    "Representative Misclassified Predictions",
    "representative_misclassified_predictions.pdf",
    "red"
)

print("Representative prediction figures saved.")


# Comprehensive balanced representative predictions

import numpy as np
import matplotlib.pyplot as plt
import os

all_images = []
all_labels = []

for x, y in test_data:
    all_images.append(x.numpy())
    all_labels.append(y.numpy())

all_images = np.concatenate(all_images, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

pred_probs = mobilenet_model.predict(all_images)
pred_labels = np.argmax(pred_probs, axis=1)
confidence = np.max(pred_probs, axis=1)

display_names = {
    "downdog": "Downward Dog",
    "goddess": "Goddess",
    "plank": "Plank",
    "tree": "Tree",
    "warrior2": "Warrior II"
}

selected_indices = []

# Select 3 strongest correct predictions per class
for class_id, class_name in enumerate(class_names):
    class_correct = np.where(
        (all_labels == class_id) & (pred_labels == class_id)
    )[0]

    class_correct = sorted(
        class_correct,
        key=lambda idx: confidence[idx],
        reverse=True
    )

    selected_indices.extend(class_correct[:3])

# Plot 15 images: 3 per class
plt.figure(figsize=(15, 12))

for i, idx in enumerate(selected_indices):
    plt.subplot(5, 3, i + 1)

    plt.imshow(all_images[idx].astype("uint8"))
    plt.axis("off")

    true_label = display_names[class_names[int(all_labels[idx])]]
    pred_label = display_names[class_names[int(pred_labels[idx])]]
    conf = confidence[idx] * 100

    plt.title(
        f"True: {true_label}\nPred: {pred_label}\nConfidence: {conf:.1f}%",
        color="green",
        fontsize=9
    )

plt.suptitle(
    "Comprehensive Representative Correct Predictions Across All Yoga Poses",
    fontsize=15,
    fontweight="bold"
)

plt.tight_layout(rect=[0, 0, 1, 0.96])

plt.savefig(
    os.path.join(FIGURE_DIR, "comprehensive_correct_predictions.pdf"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Comprehensive correct prediction figure saved.")


# ============================================================
# Detailed Mixed Prediction Analysis Figure
# Correct + Misclassified + Confidence Scores
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import os

all_images = []
all_labels = []

for x, y in test_data:
    all_images.append(x.numpy())
    all_labels.append(y.numpy())

all_images = np.concatenate(all_images, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

pred_probs = mobilenet_model.predict(all_images)
pred_labels = np.argmax(pred_probs, axis=1)
confidence = np.max(pred_probs, axis=1)

display_names = {
    "downdog": "Downward Dog",
    "goddess": "Goddess",
    "plank": "Plank",
    "tree": "Tree",
    "warrior2": "Warrior II"
}

correct_indices = np.where(pred_labels == all_labels)[0]
wrong_indices = np.where(pred_labels != all_labels)[0]

# Pick high-confidence correct predictions
correct_sorted = sorted(
    correct_indices,
    key=lambda idx: confidence[idx],
    reverse=True
)

# Pick high-confidence wrong predictions for useful error analysis
wrong_sorted = sorted(
    wrong_indices,
    key=lambda idx: confidence[idx],
    reverse=True
)

# Mixed selection: 10 correct + 10 wrong
selected_indices = correct_sorted[:10] + wrong_sorted[:10]

plt.figure(figsize=(16, 16))

for i, idx in enumerate(selected_indices):
    ax = plt.subplot(5, 4, i + 1)

    ax.imshow(all_images[idx].astype("uint8"))
    ax.set_xticks([])
    ax.set_yticks([])

    true_label = display_names[class_names[int(all_labels[idx])]]
    pred_label = display_names[class_names[int(pred_labels[idx])]]
    conf = confidence[idx] * 100

    is_correct = int(all_labels[idx]) == int(pred_labels[idx])

    border_color = "green" if is_correct else "red"
    status = "Correct" if is_correct else "Misclassified"

    for spine in ax.spines.values():
        spine.set_edgecolor(border_color)
        spine.set_linewidth(4)

    ax.set_title(
        f"{status}\nTrue: {true_label}\nPred: {pred_label}\nConf: {conf:.1f}%",
        color=border_color,
        fontsize=8
    )

plt.suptitle(
    "Detailed Mixed Prediction Analysis of MobileNetV2",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout(rect=[0, 0, 1, 0.96])

plt.savefig(
    os.path.join(FIGURE_DIR, "detailed_mixed_prediction_analysis.pdf"),
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Detailed mixed prediction analysis figure saved.")


# ============================================================
# Section 8: Model Comparison and Class-wise F1 Score Figures
# ============================================================

# Evaluate Custom CNN and MobileNetV2 on test set
custom_test_loss, custom_test_acc = custom_cnn.evaluate(test_data, verbose=0)
mobile_test_loss, mobile_test_acc = mobilenet_model.evaluate(test_data, verbose=0)

comparison_df = pd.DataFrame({
    "Model": ["Custom CNN", "MobileNetV2"],
    "Test Accuracy": [custom_test_acc, mobile_test_acc],
    "Test Loss": [custom_test_loss, mobile_test_loss]
})

comparison_df.to_excel(
    os.path.join(RESULT_DIR, "model_comparison.xlsx"),
    index=False
)

plt.figure(figsize=(7,5))
bars = plt.bar(
    comparison_df["Model"],
    comparison_df["Test Accuracy"] * 100
)

plt.ylabel("Test Accuracy (%)")
plt.title("Model Performance Comparison")
plt.ylim(0, 100)

for bar in bars:
    value = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        value + 1,
        f"{value:.1f}%",
        ha="center"
    )

plt.tight_layout()
plt.savefig(
    os.path.join(FIGURE_DIR, "model_performance_comparison.pdf"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()

# Class-wise F1 scores from MobileNetV2 report
f1_df = report_df.loc[class_names, ["f1-score"]].reset_index()
f1_df.columns = ["Class", "F1 Score"]

f1_df.to_excel(
    os.path.join(RESULT_DIR, "classwise_f1_scores.xlsx"),
    index=False
)

plt.figure(figsize=(8,5))
bars = plt.bar(f1_df["Class"], f1_df["F1 Score"])

plt.ylabel("F1 Score")
plt.xlabel("Yoga Pose Class")
plt.title("Class-wise F1 Scores for MobileNetV2")
plt.ylim(0, 1)

for bar in bars:
    value = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        value + 0.02,
        f"{value:.3f}",
        ha="center"
    )

plt.tight_layout()
plt.savefig(
    os.path.join(FIGURE_DIR, "classwise_f1_scores.pdf"),
    dpi=300,
    bbox_inches="tight"
)
plt.show()

print("Model comparison and F1-score figures saved.")
print(comparison_df)


import os
import shutil

BASE = "/kaggle/working/Yoga-Pose-Recognition"
TARGET = "/kaggle/working/Yoga-Pose-Recognition-GitHub"

# Create clean target structure
folders = ["Figures_2", "Models", "Results", "Source", "Notebook"]
for f in folders:
    os.makedirs(os.path.join(TARGET, f), exist_ok=True)

# Copy files
for folder in folders:
    src_folder = os.path.join(BASE, folder)
    dst_folder = os.path.join(TARGET, folder)
    if os.path.exists(src_folder):
        for file in os.listdir(src_folder):
            shutil.copy2(os.path.join(src_folder, file), dst_folder)

# Copy README
shutil.copy2(os.path.join(BASE, "README.md"), os.path.join(TARGET, "README.md"))

print(f"GitHub-ready folder created at {TARGET}")


import os

BASE = "/kaggle/working/Yoga-Pose-Recognition"

for root, dirs, files in os.walk(BASE):
    print("\n", root)
    for f in files:
        print("   ", f)


readme_text = """
# Yoga Pose Recognition Using Deep Learning and Transfer Learning

## Project Overview

This project develops an automated yoga pose recognition system using:

- Custom CNN
- MobileNetV2 Transfer Learning

The system classifies:

- Downward Dog
- Goddess
- Plank
- Tree
- Warrior II

## Dataset

Total Images: 1550

Training Images: 1080
Testing Images: 470

## Results

| Model | Test Accuracy |
|---------|---------|
| Custom CNN | 74.68% |
| MobileNetV2 | 88.72% |

## Repository Structure

Figures_2/
Models/
Results/
Source/

## Future Work

- Real-time pose recognition
- Mobile deployment
- 107-pose yoga classification
"""

with open("/kaggle/working/Yoga-Pose-Recognition/README.md", "w") as f:
    f.write(readme_text)

print("README created.")


import shutil
import os

BASE = "/kaggle/working/Yoga-Pose-Recognition"
TARGET = "/kaggle/working/Yoga-Pose-Recognition-GitHub"

# Create target folder
os.makedirs(TARGET, exist_ok=True)

# Copy all main folders
for folder in ["Figures_2", "Models", "Results", "Source", "Notebook"]:
    src_folder = os.path.join(BASE, folder)
    dst_folder = os.path.join(TARGET, folder)
    if os.path.exists(src_folder):
        for f in os.listdir(src_folder):
            shutil.copy2(os.path.join(src_folder, f), dst_folder)

# Copy README
shutil.copy2(os.path.join(BASE, "README.md"), os.path.join(TARGET, "README.md"))

# Make ZIP
zip_path = "/kaggle/working/Yoga-Pose-Recognition-GitHub.zip"
shutil.make_archive(base_name=zip_path.replace(".zip",""), format='zip', root_dir=TARGET)

print(f"GitHub-ready ZIP created: {zip_path}")
